# Camera-ready experiments (NeurIPS 2026, submission 33333)

Runtime → Change runtime type → **GPU** (L4 or A100 preferred; T4 works but is slower).

Run the cells top to bottom. Results go to Google Drive, so a disconnect loses at most the run in progress. Re-running a group cell skips finished runs.

| Cell | What | Rough time on an L4 |
|---|---|---|
| Smoke test | Tiny version of everything, about 5 min | 5 min |
| `layernorm` | R3–R5: 5 runs × 10k steps, c_y and blocks every 25 steps | 2–4 h |
| `ntk` | R1: 10 runs, NTK + KTA every 1000 steps | 1–2 h |
| `ntk_convergence` | R1: NTK vs. number of samples | ~5 min |
| `adamw` | R2: 4 full-batch AdamW runs | 1–2 h |

If you can open several Colab sessions, run `layernorm`, `ntk` and `adamw` in parallel: each session runs setup and then one group.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If the repo is private, use https://<GITHUB_TOKEN>@github.com/lutscha/slt-experiments.git
!git clone -b camera-ready https://github.com/lutscha/slt-experiments.git /content/slt-experiments || (cd /content/slt-experiments && git pull)
%cd /content/slt-experiments
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.environ['DATASETS'] = '/content/datasets'
os.environ['RESULTS'] = '/content/drive/MyDrive/slt_camera_ready'
os.makedirs(os.environ['DATASETS'], exist_ok=True)
os.makedirs(os.environ['RESULTS'], exist_ok=True)

## 1. Smoke test (results go to /content/smoke, not Drive)

In [ ]:
!RESULTS=/content/smoke python camera_ready/run.py layernorm --smoke --only 1
!RESULTS=/content/smoke python camera_ready/run.py ntk --smoke --only 0
!RESULTS=/content/smoke python camera_ready/run.py ntk_convergence --smoke
!RESULTS=/content/smoke python camera_ready/run.py adamw --smoke --only 1
!RESULTS=/content/smoke python camera_ready/analyze.py --smoke

## 2. Experiments

In [ ]:
!python camera_ready/run.py layernorm 2>&1 | tee -a $RESULTS/log_layernorm.txt | grep -E '===|skip|c_y|Traceback|Error'

In [ ]:
!python camera_ready/run.py ntk 2>&1 | tee -a $RESULTS/log_ntk.txt | grep -E '===|skip|ntk|Traceback|Error'

In [ ]:
!python camera_ready/run.py ntk_convergence 2>&1 | tee -a $RESULTS/log_ntk_convergence.txt

In [ ]:
!python camera_ready/run.py adamw 2>&1 | tee -a $RESULTS/log_adamw.txt | grep -E '===|skip|Preconditioned|Traceback|Error'

## 3. Analysis (CSVs and quick-look plots in `$RESULTS/camera_ready_out`)

In [ ]:
!python camera_ready/analyze.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(os.environ['RESULTS'] + '/camera_ready_out/*.png')):
    print(f); display(Image(f))